In [2]:
from pathlib import Path
import pandas as pd
from rdflib import Graph

In [3]:
# --- paths (same approach as your class notebook) ---
NB_DIR = Path.cwd().resolve()
SRC_DIR = None
for p in [NB_DIR] + list(NB_DIR.parents):
    if (p / "assignment" / "src").exists():
        SRC_DIR = p / "assignment" / "src"; break
    if (p / "src").exists():
        SRC_DIR = p / "src"; break
if SRC_DIR is None:
    raise FileNotFoundError(f"No 'src' directory above {NB_DIR}")
DATA_DIR = SRC_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
TTL = {
    "bfo":  SRC_DIR / "bfo-core.ttl",
    "ies":  SRC_DIR / "ies.ttl",
    "qudt": SRC_DIR / "qudt.ttl",
    "ccom": SRC_DIR / "ccom.ttl",
    "ccot": SRC_DIR / "ccot.ttl",
    "to":   SRC_DIR / "time.ttl",
}


In [5]:
AXIOM_QUERY = """
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?iri ?label ?axiom WHERE {
  VALUES ?type { owl:Class rdfs:Class }
  ?iri a ?type .
  FILTER(isIRI(?iri))
  FILTER NOT EXISTS { ?iri owl:deprecated ?d . FILTER(LCASE(STR(?d)) = "true") }
  OPTIONAL { ?iri rdfs:label ?label }
  FILTER(!BOUND(?label) || !CONTAINS(LCASE(STR(?label)), "deprecated"))

  {
    ?iri rdfs:subClassOf ?r .
    ?r a owl:Restriction ; owl:onProperty ?p ; ?kind ?filler .
    FILTER(?kind IN (owl:someValuesFrom, owl:allValuesFrom, owl:hasValue,
                     owl:minCardinality, owl:maxCardinality, owl:cardinality,
                     owl:minQualifiedCardinality, owl:maxQualifiedCardinality,
                     owl:qualifiedCardinality))
    BIND(CONCAT("SubClassOf ", STR(?p), " ", REPLACE(STR(?kind), "^.*#", ""),
                " ", STR(?filler)) AS ?axiom)
  } UNION {
    ?iri rdfs:subClassOf ?sup .
    FILTER(isIRI(?sup))
    BIND(CONCAT("SubClassOf ", STR(?sup)) AS ?axiom)
  } UNION {
    ?iri owl:equivalentClass ?eq .
    BIND(CONCAT("EquivalentTo ", STR(?eq)) AS ?axiom)
  }
}
ORDER BY ?iri
"""

for key, path in TTL.items():
    g = Graph()
    g.parse(path)
    rows = [{
        "iri": str(r.iri),
        "label": str(r.label) if r.label is not None else None,
        "axiom": str(r.axiom),
    } for r in g.query(AXIOM_QUERY)]
    df = pd.DataFrame(rows, columns=["iri", "label", "axiom"])
    out = DATA_DIR / f"{key}-axioms.xlsx"
    df.to_excel(out, index=False)
    print(f"✅ {key}: {len(df)} rows → {out.name}")

✅ bfo: 66 rows → bfo-axioms.xlsx
✅ ies: 574 rows → ies-axioms.xlsx
✅ qudt: 222 rows → qudt-axioms.xlsx
✅ ccom: 27 rows → ccom-axioms.xlsx
✅ ccot: 49 rows → ccot-axioms.xlsx
✅ to: 114 rows → to-axioms.xlsx
